# Engineering-First: Faceless Developer Pipeline
This notebook allows you to test the document processing logic directly, without the FastAPI or HTML frontend layer. It is perfect for debugging prompts, testing new models, and saving outputs directly to the Relational CSV database.

In [10]:
import os
import json
from src.engines.vlm import VLMEngine
from src.engines.llm import LLMEngine
from src.core.document_processor import DocumentProcessor
from src.utils.csv_db import CSVDatabase

# Ensure vLLM servers are running on ports 8000 (VLM) and 8001 (LLM) before initializing
vlm_engine = VLMEngine(server_url="http://localhost:8000/v1", model_name="PaddlePaddle/PaddleOCR-VL")
llm_engine = LLMEngine(server_url="http://localhost:8001/v1", model_name="Qwen3-4B-AWQ")

doc_processor = DocumentProcessor(vlm_engine, llm_engine)
db = CSVDatabase(data_dir="data")
print("Engines & Database Initialized!")

Creating model: ('PP-DocLayoutV3', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/teamspace/studios/this_studio/.paddlex/official_models/PP-DocLayoutV3`.


--- Initializing PaddleOCRVL (Backend: http://localhost:8000/v1) ---


Creating model: ('PaddleOCR-VL-1.5-0.9B', None)


--- VLM Initialized ---
Engines & Database Initialized!


## 1. Run Pipeline on a Test Document

In [11]:
test_file_path = "invoices/demo3.jpeg" # UPDATE THIS PATH TO A REAL PDF OR IMAGE
temp_dir = "temp/dev_pipeline"

if not os.path.exists(test_file_path):
    print(f"⚠️ Please update the path! '{test_file_path}' does not exist.")
else:
    print(f"Processing {test_file_path}...")
    diagnostic_pages = doc_processor.process(test_file_path, temp_dir)
    print(f"\n✅ Processed {len(diagnostic_pages)} pages.")

Processing invoices/demo3.jpeg...

✅ Processed 1 pages.


## 2. Inspect 'Glass Pipeline' Intermediates (Page 1 Example)

In [12]:
if 'diagnostic_pages' in locals() and diagnostic_pages:
    page_1 = diagnostic_pages[0]
    print("=================== VLM RAW OUTPUT ===================")
    print(page_1.get('stage_1_vlm_raw', ''))
    print("\n=================== CLEANED TEXT ===================")
    print(page_1.get('stage_2_cleaned', ''))
    print("\n=================== LLM JSON ===================")
    print(json.dumps(page_1.get('stage_3_llm_json', {}), indent=2))

=================== VLM RAW OUTPUT ===================

Photography
Invoice
123 Street Name
Denver, CO 80205
P: 555-555-5555
email@samplebusiness.com
INVOICE
Invoice #: 12074
Invoice date: 7/19/24
Job Details: Photo Shoot
Bill to: Customer Name
Address: 123 Street Name Denver, CO 80205
Phone: 555-555-5555
<table><tr><td>Description</td><td>Qty</td><td>Unit price</td><td>Discount</td><td>Price</td></tr><tr><td>Wedding Ceremony Photos</td><td>1</td><td>$1,500.00</td><td></td><td>$1,500.00</td></tr><tr><td>Bride &amp; Groom Portraits</td><td>1</td><td>$500.00</td><td></td><td>$500.00</td></tr><tr><td>Engagement Photoshoot</td><td>1</td><td>$300.00</td><td></td><td>$300.00</td></tr><tr><td></td><td></td><td></td><td></td><td>$0.00</td></tr><tr><td></td><td></td><td></td><td></td><td>$0.00</td></tr><tr><td></td><td></td><td></td><td></td><td>$0.00</td></tr><tr><td></td><td></td><td></td><td></td><td>$0.00</td></tr><tr><td></td><td></td><td></td><td></td><td>$0.00</td></tr><tr><td></td><td><

## 3. Save to Relational CSVs

In [6]:
if 'diagnostic_pages' in locals() and diagnostic_pages:
    invoice_id = db.save_extraction(diagnostic_pages)
    print(f"✅ Saved successfully! Invoice ID: {invoice_id}")
    print("Check the 'data/invoices.csv' and 'data/line_items.csv' files.")

✅ Saved successfully! Invoice ID: 59eed633-55bf-4d2a-9eb6-14c41fc99d93
Check the 'data/invoices.csv' and 'data/line_items.csv' files.
